# Phase A — Raw eventRows 시각화

ai-feat-294. 통계 metric 한계 도달 후 raw eventRows 를 사람 눈으로 직접 보고 패턴을 발굴하기 위한 도구.

**산출물**
- 셀 4: 그룹별 카운트 + Balabit user 분포 검증 (51 / 101 / 500 기대)
- 셀 5: 단일 trial 동작 확인 (lv2_human / lv2_macro / balabit 각 1개)
- 셀 7~9: trajectory grid 3장 → `outputs/294_raw_trajectory/*.png`
- 셀 11~13: Phase B 자유 탐색 helper (단일 trial deep dive, plotly interactive)

**전제**
- `pip install -r requirements-eda.txt` 완료
- `services/ai/data/behavior/` 에 trial_*.json 652개 (lv2 152 + Balabit 500)

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.dpi"] = 100

In [ ]:
from trial_loader import (
    GROUPS,
    DATA_DIR,
    load_trial,
    list_trials_by_group,
    event_rows_to_df,
    sample_trials,
    parse_balabit_user,
)
from plots import (
    PLOT_TYPES,
    plot_trajectory,
    plot_speed_over_time,
    plot_dt_distribution,
    plot_pre_click_paths,
    plot_grid,
    plot_single_trial_interactive,
)

OUTPUT_DIR = Path("outputs") / "294_raw_trajectory"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR exists:", DATA_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

In [ ]:
# 그룹별 카운트 (기대: lv2_human=51, lv2_macro=101, balabit=500)
group_metas = {g: list_trials_by_group(g) for g in GROUPS}
for g, metas in group_metas.items():
    print(f"{g}: {len(metas)} trials")

# Balabit user 분포
balabit_users = Counter(m["user_id"] for m in group_metas["balabit"])
print("\nBalabit user 분포:")
for user, n in sorted(balabit_users.items(), key=lambda kv: (kv[0] is None, kv[0])):
    print(f"  {user}: {n}")

# session_id 포맷 sample
print("\nsession_id 샘플:")
print("  lv2  :", group_metas["lv2_human"][0]["session_id"])
print("  balabit:", group_metas["balabit"][0]["session_id"])
print("  parse_balabit_user ->", parse_balabit_user(group_metas["balabit"][0]["session_id"]))

In [ ]:
# 단일 trial 동작 확인 — 그룹별 1개씩 4 plot
sample_ids = {
    "lv2_human": group_metas["lv2_human"][0]["trial_id"],
    "lv2_macro": group_metas["lv2_macro"][0]["trial_id"],
    "balabit":   group_metas["balabit"][0]["trial_id"],
}

fig, axes = plt.subplots(3, 4, figsize=(16, 11))
for row, (group, tid) in enumerate(sample_ids.items()):
    trial = load_trial(tid)
    plot_trajectory(trial,        ax=axes[row, 0])
    plot_speed_over_time(trial,   ax=axes[row, 1])
    plot_dt_distribution(trial,   ax=axes[row, 2])
    plot_pre_click_paths(trial,   ax=axes[row, 3])
    axes[row, 0].set_ylabel(f"{group}\n{axes[row, 0].get_ylabel()}", fontsize=9)
fig.suptitle("4 plots × 3 groups (single trial each)", fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

## Grid views — 그룹 비교

각 그룹별 sample 을 grid 로 모아 한눈에 비교. trajectory 3장은 `outputs/294_raw_trajectory/` 에 PNG 저장됨 (gitignore).

In [ ]:
fig = plot_grid(
    "lv2_human", "trajectory",
    save_path=OUTPUT_DIR / "lv2_human_trajectory_grid.png",
)
plt.show()

In [ ]:
fig = plot_grid(
    "lv2_macro", "trajectory",
    save_path=OUTPUT_DIR / "lv2_macro_trajectory_grid.png",
)
plt.show()

In [ ]:
fig = plot_grid(
    "balabit", "trajectory",
    save_path=OUTPUT_DIR / "balabit_trajectory_grid.png",
)
plt.show()

## Phase B 자유 탐색 helper

Grid 에서 흥미로운 trial 을 발견하면 아래 셀에 trial_id 를 넣어 4 plot 한 번에 보거나, plotly interactive 로 deep dive.

In [ ]:
def show_trial_4plot(trial_id: int, figsize=(16, 4)):
    trial = load_trial(trial_id)
    fig, axes = plt.subplots(1, 4, figsize=figsize)
    plot_trajectory(trial,      ax=axes[0])
    plot_speed_over_time(trial, ax=axes[1])
    plot_dt_distribution(trial, ax=axes[2])
    plot_pre_click_paths(trial, ax=axes[3])
    fig.tight_layout()
    return fig

# 예시
show_trial_4plot(sample_ids["lv2_human"])
plt.show()

In [ ]:
# 다른 plot_type 의 grid 도 필요 시 활성화
# plot_grid("lv2_human", "speed");      plt.show()
# plot_grid("lv2_macro", "speed");      plt.show()
# plot_grid("balabit",   "speed");      plt.show()
# plot_grid("lv2_human", "dt");         plt.show()
# plot_grid("lv2_macro", "dt");         plt.show()
# plot_grid("balabit",   "dt");         plt.show()
# plot_grid("lv2_human", "pre_click");  plt.show()
# plot_grid("lv2_macro", "pre_click");  plt.show()
# plot_grid("balabit",   "pre_click");  plt.show()

In [ ]:
# plotly interactive — trial_id 바꿔가며 hover 로 ts_ms / event / speed 확인
fig = plot_single_trial_interactive(sample_ids["lv2_macro"])
fig.show()